# wandb-log-step — ex1: log loss + examples_seen step over a fake epoch

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-log-step`. Running the final beacon cell reports progress against the `Logging: wandb.log step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.log step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-log-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-log-step"
DD_SUBTOPIC = "Logging: wandb.log step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.log({...}, step=step_count)` — quick refresher

`wandb.log(metrics_dict, step=int)` sends one row of scalar metrics to the wandb dashboard. The `step` kwarg is the x-axis value — wandb plots `metrics` against it.

**Why pass `step` explicitly.** If you omit it, wandb uses an internal monotonic step counter that increments by 1 per `log` call. This is fine for single-runs, but disastrous when comparing runs with different batch sizes or different log frequencies — the x-axes don't line up.

**ARENA convention: step = `self.examples_seen`.** Examples-seen is batch-size-invariant: 50 000 examples is 50 000 examples whether your batch was 32 or 256. The training-step counter is NOT — step 1000 with batch=32 has seen 32 000 examples, batch=256 has seen 256 000.

**Values must be Python scalars (not tensors).** `loss.item()` not `loss`. Wandb serializes the dict to JSON; tensors aren't JSON-able. Pass `loss.item()`, `accuracy` (already a float), etc.

**Multiple metrics in one call.** `wandb.log({'loss': l, 'acc': a, 'lr': lr}, step=s)` is preferred over three separate `wandb.log` calls — one call = one row in the dashboard.

### Exercise 1 — log loss + examples_seen step over a fake epoch

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.log({...}, step=examples_seen)` inside a fake training loop with an examples-seen counter, verified by mocking the `wandb` module and inspecting every recorded call.
> Keywords: wandb, log, examples-seen, step-axis, mock
> ```

**KCs targeted:** `wandb-log-metrics-dict`, `wandb-log-step-kwarg`

Implement `ex1_fake_epoch_with_wandb_log(losses, batch_size)`. The canonical ARENA inner-loop logging recipe:

1. `losses` is a Python list of floats — one per fake training step.
2. `batch_size` is an int — examples per step.
3. Maintain a running `examples_seen` counter, starting at 0.
4. For each `loss` in `losses`:
   - Increment `examples_seen` by `batch_size` BEFORE logging.
   - Call `wandb.log({'loss': loss}, step=examples_seen)`.
5. Return the final `examples_seen` value.

The test inspects `wandb.log.call_args_list` to check every call's step value lines up with the cumulative examples-seen.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_fake_epoch_with_wandb_log(losses, batch_size):
    examples_seen = 0
    for loss in losses:
        examples_seen += batch_size
        wandb.log({'loss': loss}, step=examples_seen)
    return examples_seen


<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_fake_epoch_with_wandb_log(losses, batch_size):
    examples_seen = 0
    for loss in losses:
        examples_seen += batch_size
        wandb.log({'loss': loss}, step=examples_seen)
    return examples_seen
```

**Why increment BEFORE logging.** Step 1's log call should show step=batch_size, not step=0 (that would imply the model hasn't trained yet). The semantics are 'after this batch of batch_size examples, the loss is X'.

**Why `examples_seen`, not `step_idx`.** A run with batch=32 vs batch=256 spends very different wall-clock per step. Plotting on the examples-seen axis lets you overlay the two and see which configuration converges faster per example. Step-idx would have the batch=32 run reach step 1000 in 1/8th the wall-clock.

**The metrics dict is the FIRST positional arg.** Wandb's API accepts both `wandb.log(d, step=n)` and `wandb.log(data=d, step=n)`; ARENA uses positional. The test tolerates both.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()